1. Import the dependencies:

In [3]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI
import ollama
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)



# Ollama client wrapper (optional, for OpenAI-like usage)
class OllamaClient:
    def __init__(self, model="llama3.2"):
        self.model = model

    def chat(self, messages):
        response = ollama.chat(model=self.model, messages=messages)
        return response["message"]["content"]  # Fix incorrect response handling

# Create an Ollama client instance
ollama_client = OllamaClient()

# Test the function
# response = ollama_client.chat("What is the capital of France?")
# print("Ollama's Response:", response)


In [6]:
class Website:

    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [8]:
my_site = Website("https://www.google.fr/")
print(my_site.title)
print(my_site.text)

Google
Recherche
Images
Maps
Play
YouTube
Actualités
Gmail
Drive
Plus
»
Historique Web
|
Paramčtres
|
Connexion
Recherche avancée
Google disponible en :
العربية
Publicité
Ŕ propos de Google
Google.com
© 2025 -
Confidentialité
-
Conditions


In [ ]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [13]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [14]:
print(user_prompt_for(my_site))

You are looking at a website titled Google
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Recherche
Images
Maps
Play
YouTube
Actualités
Gmail
Drive
Plus
»
Historique Web
|
Paramčtres
|
Connexion
Recherche avancée
Google disponible en :
العربية
Publicité
Ŕ propos de Google
Google.com
© 2025 -
Confidentialité
-
Conditions


In [15]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [18]:
response = ollama_client.chat(messages)
print("Ollama's Response:", response)

Ollama's Response: *sigh* Oh, wow. I'm just so glad you came to me with this incredibly difficult math problem. It's not like it's something that every basic arithmetic textbook would teach or anything.

Fine. If you must know, the answer to your question is... (dramatic pause) ...4. Congratulations, you're a certified math genius now!


In [24]:
def messages_for(website):
    return [
        {"role" : "system", "content": system_prompt},
        {"role" : "user" , "content" : user_prompt_for(website)}
    ]

In [25]:
messages_for(my_site)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Google\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nRecherche\nImages\nMaps\nPlay\nYouTube\nActualités\nGmail\nDrive\nPlus\n»\nHistorique Web\n|\nParamčtres\n|\nConnexion\nRecherche avancée\nGoogle disponible en\xa0:\nالعربية\nPublicité\nŔ propos de Google\nGoogle.com\n© 2025 -\nConfidentialité\n-\nConditions'}]

In [26]:
def summarize(url):
    website = Website(url)
    response = ollama_client.chat(messages_for(website))
    return response

In [28]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [29]:
display_summary("https://fr.wikipedia.org/wiki/Cristiano_Ronaldo")

Cristiano Ronaldo est un footballeur portugais né le 5 février 1985 à Funchal, aux Canaries. Il est considéré comme l'un des meilleurs footballeurs de tous les temps. Voici une liste de ses réalisations et de ses honneurs :

*   **Champion du monde** : vainqueur de la Coupe du monde en 2016 avec le Portugal.
*   **Ligue des champions**: vainqueur de la Ligue des champions de l'UEFA à cinq reprises (2008, 2014, 2016, 2017 et 2018) avec les clubs Real Madrid, Manchester United, Juventus et Liverpool.
*   **Champion d'Europe** : vainqueur du Championnat d'Europe en 2016 avec le Portugal.
*   **Meilleur buteur de la Ligue des champions**: vainqueur du titre de meilleur buteur de la Ligue des champions de l'UEFA à trois reprises (2008, 2013 et 2014).
*   **Ballon d'or** : vainqueur du Ballon d'or en 2008, 2013, 2014 et 2016.
*   **Meilleur footballeur de la saison UEFA Champions League**: vainqueur de ce titre à trois reprises (2007/08, 2011/12 et 2015/16).
*   **Ligue des champions**: record détenu en championnat avec un total de 150 buts marqués.
*   **Buteurs record**: record détenu en Ligue des champions pour le nombre de buts marqués par un joueur (150).
*   **Titres de club** : vainqueur de nombreux titres de club, notamment la Supercoupe d'Espagne et la Coupe d'Afrique des nations.
*   **Récords personnels** : détient de nombreux records personnels, tels que le nombre de buts marqués en Ligue des champions (150), le nombre de buts marqués pour un club en championnat (755) et le nombre de buts marqués par un joueur en une saison (54).
*   **Sélection** : sélectionné pour la Coupe du monde de 2006, la coupe d'Europe des nations 2008-09, la coupe continentale des nations 2011-12, la Ligue des champions de l'UEFA et la Coupe du roi en 2010-11 et 2012-13.
*   **Équipe de l'an** : sélectionné pour l'équipe de l'an par le magazine FIFA en 2008, 2009, 2010 et 2011.

Cristiano Ronaldo est considéré comme l'un des meilleurs footballeurs de tous les temps. Il a remporté de nombreux titres avec ses clubs et sa sélection nationale, et il détient de nombreux records personnels et de club.